# Layer 3 — Phase 2 v3: Window-level KPI Feature Engineering

## v2 → v3 변경사항 (6개)

1. **duration 분포 검증**: timeout threshold 5초가 데이터 분포에 합리적인지 P90/P95/P99/P999 출력 + metadata 저장
2. **proxy 변수 정의 metadata**: `ingestion_delay_proxy`, `log_missing_rate`의 한계를 metadata에 명시 (발표 방어용)
3. **train_df 길이 체크**: 100분 미만이면 즉시 중단 (LSTM 학습 의미 없음)
4. **segment 기반 sequence 생성**: train sequence가 시간 불연속 구간을 가로지르지 않도록 segment별로 생성
5. **error rate fallback 명확화**: 셀 시작 시 fallback 가능성 markdown 명시 + metadata 저장
6. **risk-normalized table 별도 저장**: Part 3 rule-based score용 0~1 정규화 KPI table

## 흐름
```
Step A. 컬럼 확정
Step B. Timestamp 통일
Step C. 공통 1분 grid 생성
Step D. 결측/이상값 처리 함수 정의
Step E. KPI 추출
  E-1. P99 latency / counts / timeout (+ duration 분포 검증)
  E-2. API error rate (+ fallback)
  E-3. Ingestion delay proxy
  E-4. Log missing rate
  E-검증: 시계열 시각화
Step F. Incident label
  F-검증: 라벨 분포
Step G. KPI table 병합 + 저장
Step H. Train/Test split (±10분 buffer + 길이 체크)
Step I. 정규화 + percentile metadata + risk-normalized table 저장
Step J. 시퀀스 윈도우 생성 (segment 기반)
  J-검증: shape
```

## 0. 환경 셋업

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import glob
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = '/content/drive/MyDrive/layer3_data'
WORK = f'{ROOT}/work'
OUT  = f'{ROOT}/processed'

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

!apt -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Part 1 결과물 로드
with open(os.path.join(OUT, 'aiops_data_index.json'), 'r', encoding='utf-8') as f:
    DATA_INDEX = json.load(f)

with open(os.path.join(OUT, 'aiops_column_candidates.json'), 'r', encoding='utf-8') as f:
    COL_CANDIDATES = json.load(f)

faults = pd.read_parquet(os.path.join(OUT, 'aiops_fault_labels.parquet'))

DAILY_DIR = DATA_INDEX['aiops_daily_dir']
BIZ_DIR_NAME = DATA_INDEX['business_subdir_name']
INFRA_DIR_NAME = DATA_INDEX['infra_subdir_name']
TRACE_DIR_NAME = DATA_INDEX['trace_subdir_name']
FAULT_TIME_COL = DATA_INDEX['fault_time_col']
DAYS = DATA_INDEX['aiops_days_extracted']

print(f'풀린 일자: {DAYS}')
print(f'business 폴더명: {BIZ_DIR_NAME}')
print(f'infra 폴더명: {INFRA_DIR_NAME}')
print(f'trace 폴더명: {TRACE_DIR_NAME}')
print(f'fault 시간 컬럼: {FAULT_TIME_COL}')
print(f'fault 라벨 행 수: {len(faults)}')

## Step A. 컬럼 확정

⚠️ **여기가 가장 막힐 가능성 큼.** 추정 작성, 실제 컬럼명에 맞게 수정 필수.

In [ ]:
# 컬럼 후보 출력 (참고용)
for src in ['business', 'trace', 'fault_label']:
    if src in COL_CANDIDATES:
        print(f'\n=== {src.upper()} columns ===')
        print(COL_CANDIDATES[src]['columns'])
        print(f'\nkeyword 매칭:')
        for kw, cols in COL_CANDIDATES[src]['keyword_matches'].items():
            print(f'  [{kw}] → {cols}')

In [ ]:
# 👇 위 출력 보고 실제 컬럼명으로 수정 필수

# === Trace ===
TRACE_TIME_COL = None       # 예: 'startTime', 'timestamp'
TRACE_DURATION_COL = None   # 예: 'elapsedTime', 'duration'
TRACE_TIME_UNIT = 'ms'
TRACE_TIMESTAMP_UNIT = 'ms'

# === Business ===
BIZ_TIME_COL = None
BIZ_VALUE_COL = None
BIZ_NAME_COL = None         # 옵션
BIZ_SUCCESS_KEYWORD = 'success'

# === Fault label ===
FAULT_END_COL = None        # 옵션
FAULT_DURATION_DEFAULT = 5

# === 임계치 ===
TIMEOUT_THRESHOLD_MS = 5000

# === Buffer ===
TRAIN_BUFFER_MIN = 10

# === v3: train 최소 길이 (안전장치) ===
MIN_TRAIN_MINUTES = 100   # 100분 미만이면 LSTM 학습 의미 없음

# 검증
missing = [n for n, v in [
    ('TRACE_TIME_COL', TRACE_TIME_COL),
    ('TRACE_DURATION_COL', TRACE_DURATION_COL),
    ('BIZ_TIME_COL', BIZ_TIME_COL),
    ('BIZ_VALUE_COL', BIZ_VALUE_COL),
] if v is None]

if missing:
    raise ValueError(
        f'컬럼명 미설정: {missing}. 위 셀 출력 보고 직접 입력하세요.\n'
        f'⚠️ 한자/특수문자 정확히 복사. duration 단위 확인 (보통 ms).'
    )
print('컬럼 매핑 완료')

## Step B. Timestamp 통일

In [ ]:
def to_datetime_safe(series, hint_unit=None):
    if pd.api.types.is_numeric_dtype(series):
        sample = series.dropna().iloc[0]
        if hint_unit == 'ms' or sample > 1e12:
            return pd.to_datetime(series, unit='ms', errors='coerce')
        else:
            return pd.to_datetime(series, unit='s', errors='coerce')
    else:
        return pd.to_datetime(series, errors='coerce')

faults[FAULT_TIME_COL] = pd.to_datetime(faults[FAULT_TIME_COL], errors='coerce')
if FAULT_END_COL and FAULT_END_COL in faults.columns:
    faults[FAULT_END_COL] = pd.to_datetime(faults[FAULT_END_COL], errors='coerce')

print(f'fault start 파싱 성공: {faults[FAULT_TIME_COL].notna().sum()}/{len(faults)}')

## Step C. 공통 1분 grid 생성

In [ ]:
common_grids = {}
for d_str in DAYS:
    d = pd.to_datetime(d_str.replace('_', '-'))
    grid = pd.date_range(d, d + pd.Timedelta(days=1) - pd.Timedelta(minutes=1), freq='1min')
    common_grids[d_str] = grid
    print(f'{d_str}: {len(grid)} 분')

ALL_GRID = pd.DatetimeIndex(
    np.concatenate([common_grids[d].values for d in sorted(common_grids.keys())])
)
print(f'\n전체 grid: {len(ALL_GRID)} 포인트')

## Step D. 결측/이상값 처리 함수

In [ ]:
def clean_numeric(series, lower=None, upper=None, name='value'):
    """숫자 변환 + 하한/상한 밖 값 NaN 처리. (보간은 호출 측에서)"""
    s = pd.to_numeric(series, errors='coerce')
    n_before = s.notna().sum()
    if lower is not None:
        s = s.where(s >= lower)
    if upper is not None:
        s = s.where(s <= upper)
    n_after = s.notna().sum()
    if n_before - n_after > 0:
        print(f'  [{name}] 이상값 NaN 처리: {n_before - n_after}개')
    return s

def remove_invalid_timestamps(df, ts_col):
    n_before = len(df)
    df = df[df[ts_col].notna()].copy()
    n_removed = n_before - len(df)
    if n_removed > 0:
        print(f'  invalid timestamp 행 제거: {n_removed}개')
    return df

## Step E-1. P99 latency + counts + timeout (v3: duration 분포 검증)

**v3 추가**: TIMEOUT_THRESHOLD_MS=5000이 데이터 분포에 합리적인지 검증.

보고서 표현: *"timeout threshold는 5초를 기본 기준으로 설정하되, trace duration의 상위 분위수를 함께 확인하여 임계치가 데이터 분포와 어긋나지 않는지 검증하였다."*

In [ ]:
def load_traces_for_day(day_str):
    trace_dir = os.path.join(DAILY_DIR, day_str, TRACE_DIR_NAME)
    if not os.path.exists(trace_dir):
        print(f'⚠️ trace 폴더 없음: {trace_dir}')
        return None
    files = sorted(os.listdir(trace_dir))
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(os.path.join(trace_dir, f),
                            usecols=[TRACE_TIME_COL, TRACE_DURATION_COL])
            dfs.append(df)
        except Exception as e:
            print(f'  스킵 {f}: {e}')
    if not dfs:
        return None
    combined = pd.concat(dfs, ignore_index=True)
    print(f'  {day_str}: trace {len(combined):,} 행')
    return combined

In [ ]:
p99_list = []
request_count_list = []
timeout_count_list = []
all_durations_for_check = []  # v3: 분포 검증용

for day_str in DAYS:
    print(f'\n--- {day_str} ---')
    trace_df = load_traces_for_day(day_str)
    if trace_df is None:
        continue
    
    trace_df[TRACE_TIME_COL] = to_datetime_safe(trace_df[TRACE_TIME_COL], TRACE_TIMESTAMP_UNIT)
    trace_df = remove_invalid_timestamps(trace_df, TRACE_TIME_COL)
    trace_df[TRACE_DURATION_COL] = clean_numeric(
        trace_df[TRACE_DURATION_COL], lower=0, upper=600000, name='duration'
    )
    trace_df = trace_df.dropna(subset=[TRACE_DURATION_COL])
    trace_df['ts_min'] = trace_df[TRACE_TIME_COL].dt.floor('1min')
    
    # v3: duration 샘플 (메모리 절약 위해 최대 100k)
    sample = trace_df[TRACE_DURATION_COL].sample(
        min(100000, len(trace_df)), random_state=42
    )
    all_durations_for_check.append(sample)
    
    grouped = trace_df.groupby('ts_min')[TRACE_DURATION_COL]
    p99_list.append(grouped.quantile(0.99))
    request_count_list.append(grouped.count())
    timeout_count_list.append(
        trace_df.groupby('ts_min').apply(
            lambda x: (x[TRACE_DURATION_COL] > TIMEOUT_THRESHOLD_MS).sum()
        )
    )

p99_latency = pd.concat(p99_list).sort_index() if p99_list else pd.Series(dtype=float)
request_count = pd.concat(request_count_list).sort_index() if request_count_list else pd.Series(dtype=float)
timeout_count = pd.concat(timeout_count_list).sort_index() if timeout_count_list else pd.Series(dtype=float)
timeout_rate = (timeout_count / request_count.replace(0, np.nan)).fillna(0)

print(f'\n=== KPI 추출 결과 ===')
print(f'P99 latency: {len(p99_latency)} 분, 평균 {p99_latency.mean():.1f} ms')
print(f'request_count: 평균 {request_count.mean():.1f}건/분')
print(f'timeout_count: 평균 {timeout_count.mean():.2f}건/분')
print(f'timeout_rate: 평균 {timeout_rate.mean():.4f}')

In [ ]:
# v3: duration 분포 검증
if all_durations_for_check:
    all_durations = pd.concat(all_durations_for_check)
    duration_quantiles = all_durations.quantile([0.50, 0.75, 0.90, 0.95, 0.99, 0.999])
    
    print('=== Trace duration 분위수 (ms) ===')
    display(duration_quantiles)
    
    p99_val = float(duration_quantiles.loc[0.99])
    p999_val = float(duration_quantiles.loc[0.999])
    
    print(f'\n현재 TIMEOUT_THRESHOLD_MS = {TIMEOUT_THRESHOLD_MS}')
    print(f'데이터 P99 = {p99_val:.0f} ms')
    print(f'데이터 P999 = {p999_val:.0f} ms')
    
    # 합리성 체크
    if TIMEOUT_THRESHOLD_MS < p99_val:
        print(f'\n⚠️ 경고: timeout threshold가 P99보다 작습니다.')
        print(f'   너무 많은 정상 요청이 timeout으로 분류될 수 있어요.')
        print(f'   고려: TIMEOUT_THRESHOLD_MS를 {int(p99_val * 1.5)} 이상으로 상향')
    elif TIMEOUT_THRESHOLD_MS > p999_val * 10:
        print(f'\n⚠️ 경고: timeout threshold가 P999의 10배 이상입니다.')
        print(f'   거의 timeout이 발생하지 않을 수 있어요.')
    else:
        print(f'\n✓ timeout threshold가 합리적 범위 (P99~P999*10 사이).')
    
    # metadata 저장용
    DURATION_QUANTILES = {f'p{int(q*1000)/10}': float(v) 
                         for q, v in duration_quantiles.items()}
else:
    DURATION_QUANTILES = {}
    print('⚠️ duration 데이터 없음. 검증 스킵.')

## Step E-2. API error rate (v3: fallback 명확화)

**v3 명시**: business 데이터에서 명시적 success/error KPI를 찾지 못하면 timeout_rate를 proxy로 사용.

보고서 표현: *"business metric에서 명시적 success/error KPI가 없는 경우, timeout rate를 API error proxy로 사용하였다. 이 경우 error rate와 timeout rate는 동일한 정보를 담으므로 모델 input의 정보량이 줄어드는 한계가 있다."*

In [ ]:
def load_business_for_day(day_str):
    biz_dir = os.path.join(DAILY_DIR, day_str, BIZ_DIR_NAME)
    if not os.path.exists(biz_dir):
        return None
    files = sorted(os.listdir(biz_dir))
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(os.path.join(biz_dir, f))
            dfs.append(df)
        except Exception as e:
            print(f'  스킵 {f}: {e}')
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)

In [ ]:
error_rate_list = []

for day_str in DAYS:
    print(f'\n--- {day_str} ---')
    biz_df = load_business_for_day(day_str)
    if biz_df is None:
        print(f'  business 데이터 없음')
        continue
    print(f'  business {len(biz_df):,} 행 로드')
    
    biz_df[BIZ_TIME_COL] = to_datetime_safe(biz_df[BIZ_TIME_COL])
    biz_df = remove_invalid_timestamps(biz_df, BIZ_TIME_COL)
    
    if BIZ_NAME_COL and BIZ_NAME_COL in biz_df.columns:
        success_mask = biz_df[BIZ_NAME_COL].astype(str).str.contains(
            BIZ_SUCCESS_KEYWORD, case=False, na=False
        )
        success_df = biz_df[success_mask].copy()
        if len(success_df) == 0:
            print(f'  ⚠️ success keyword "{BIZ_SUCCESS_KEYWORD}"로 0행')
            print(f'     사용 가능 KPI: {biz_df[BIZ_NAME_COL].unique()[:10]}')
            continue
    else:
        success_df = biz_df.copy()
    
    success_df[BIZ_VALUE_COL] = clean_numeric(
        success_df[BIZ_VALUE_COL], lower=0, upper=1, name='success_rate'
    )
    success_df['ts_min'] = success_df[BIZ_TIME_COL].dt.floor('1min')
    avg_success = success_df.groupby('ts_min')[BIZ_VALUE_COL].mean()
    error_rate_list.append(1 - avg_success)

error_rate = pd.concat(error_rate_list).sort_index() if error_rate_list else pd.Series(dtype=float)

USED_FALLBACK_FOR_ERROR_RATE = False
if len(error_rate) == 0:
    print('\n⚠️ FALLBACK: business에서 error rate 추출 실패 → timeout_rate를 proxy로 사용')
    print('   보고서 명시 필요: error rate와 timeout rate가 동일 정보를 담음 (모델 input 정보량 감소 한계)')
    error_rate = timeout_rate.copy()
    USED_FALLBACK_FOR_ERROR_RATE = True

print(f'\nerror rate: {len(error_rate)} 분, 평균 {error_rate.mean():.4f}')
print(f'fallback 사용: {USED_FALLBACK_FOR_ERROR_RATE}')

## Step E-3. Ingestion delay proxy

⚠️ **명명 주의**: 실제 sensor-to-cloud delay가 아니라 *trace 부재 여부* proxy.

**v3 보고서 표현 (보수적)**: *"공개 AIOps 데이터에는 실제 sensor-to-cloud ingestion delay가 직접 포함되지 않으므로, 1분 슬롯 내 trace 부재 여부를 telemetry availability gap proxy (변수명: ingestion_delay_proxy)로 정의하였다. 정상 시간대에 요청 자체가 없는 구간도 1로 마킹될 수 있는 한계가 있다."*

In [ ]:
ingestion_delay_proxy_list = []

for day_str in DAYS:
    trace_df = load_traces_for_day(day_str)
    if trace_df is None:
        continue
    trace_df[TRACE_TIME_COL] = to_datetime_safe(trace_df[TRACE_TIME_COL], TRACE_TIMESTAMP_UNIT)
    trace_df = remove_invalid_timestamps(trace_df, TRACE_TIME_COL)
    trace_df['ts_min'] = trace_df[TRACE_TIME_COL].dt.floor('1min')
    
    counts = trace_df.groupby('ts_min').size()
    full_grid = common_grids[day_str]
    counts_full = counts.reindex(full_grid, fill_value=0)
    
    proxy = (counts_full == 0).astype(float)
    ingestion_delay_proxy_list.append(proxy)

ingestion_delay_proxy = (
    pd.concat(ingestion_delay_proxy_list).sort_index() 
    if ingestion_delay_proxy_list else pd.Series(dtype=float)
)
print(f'ingestion_delay_proxy: {len(ingestion_delay_proxy)} 분')
print(f'  trace 부재 비율: {ingestion_delay_proxy.mean():.4f}')

## Step E-4. Log missing rate (proxy)

**v3 보고서 표현 (보수적)**: *"monitoring log missing rate는 원본 로그의 실제 누락 라벨이 아니므로, business metric의 expected reporting frequency 대비 timestamp gap을 기반으로 산출한 proxy로 정의하였다."*

In [ ]:
log_missing_list = []

for day_str in DAYS:
    biz_df = load_business_for_day(day_str)
    if biz_df is None:
        continue
    biz_df[BIZ_TIME_COL] = to_datetime_safe(biz_df[BIZ_TIME_COL])
    biz_df = remove_invalid_timestamps(biz_df, BIZ_TIME_COL)
    biz_df['ts_min'] = biz_df[BIZ_TIME_COL].dt.floor('1min')
    
    counts = biz_df.groupby('ts_min').size()
    full_grid = common_grids[day_str]
    counts_full = counts.reindex(full_grid, fill_value=0)
    
    expected = counts_full[counts_full > 0].median()
    if pd.isna(expected) or expected == 0:
        expected = 1
    
    missing_rate = (1 - counts_full / expected).clip(lower=0, upper=1)
    log_missing_list.append(missing_rate)

log_missing_rate = (
    pd.concat(log_missing_list).sort_index() 
    if log_missing_list else pd.Series(dtype=float)
)
print(f'log_missing_rate: {len(log_missing_rate)} 분, 평균 {log_missing_rate.mean():.4f}')

## E-검증. KPI 시계열 시각화

In [ ]:
kpis_so_far = {
    'P99 latency (ms)': p99_latency,
    'Request count': request_count,
    'Timeout count': timeout_count,
    'Timeout rate': timeout_rate,
    'API error rate': error_rate,
    'Ingestion delay proxy': ingestion_delay_proxy,
    'Log missing rate': log_missing_rate,
}

fig, axes = plt.subplots(len(kpis_so_far), 1, figsize=(14, 2 * len(kpis_so_far)), sharex=True)
for ax, (name, series) in zip(axes, kpis_so_far.items()):
    if len(series) > 0:
        ax.plot(series.index, series.values, linewidth=0.7, color='steelblue')
        ax.set_title(f'{name}  (n={len(series)}, mean={series.mean():.4f})')
    else:
        ax.set_title(f'{name}  (empty)')
plt.tight_layout()
plt.show()

stats = pd.DataFrame({
    name: [s.count(), s.mean(), s.std(), s.min(), s.max()]
    for name, s in kpis_so_far.items()
}, index=['count', 'mean', 'std', 'min', 'max']).T
display(stats)

## Step F. Incident label 생성

In [ ]:
incident_flag = pd.Series(0, index=ALL_GRID, dtype=int)

n_faults_in_grid = 0
for _, row in faults.iterrows():
    start = row[FAULT_TIME_COL]
    if pd.isna(start):
        continue
    if FAULT_END_COL and FAULT_END_COL in faults.columns and pd.notna(row[FAULT_END_COL]):
        end = row[FAULT_END_COL]
    else:
        end = start + pd.Timedelta(minutes=FAULT_DURATION_DEFAULT)
    
    mask = (incident_flag.index >= start.floor('1min')) & \
           (incident_flag.index <= end.ceil('1min'))
    if mask.sum() > 0:
        incident_flag.loc[mask] = 1
        n_faults_in_grid += 1

print(f'fault 라벨 처리: {n_faults_in_grid}/{len(faults)}개 grid 내 마킹')
print(f'incident 분: {incident_flag.sum()} / {len(incident_flag)} ({incident_flag.mean():.4f})')

### F-검증

In [ ]:
fig, ax = plt.subplots(figsize=(14, 2))
ax.fill_between(incident_flag.index, 0, incident_flag.values, color='crimson', alpha=0.5)
ax.set_title(f'Incident flag  (총 {incident_flag.sum()}분)')
ax.set_yticks([0, 1])
plt.tight_layout()
plt.show()

fault_minutes_per_day = incident_flag.groupby(incident_flag.index.date).sum()
print('일별 fault 분 수:')
display(fault_minutes_per_day)

## Step G. KPI table 병합 + 저장

In [ ]:
kpi_table = pd.DataFrame(index=ALL_GRID)

# Model features
kpi_table['p99_latency'] = p99_latency.reindex(ALL_GRID)
kpi_table['timeout_rate'] = timeout_rate.reindex(ALL_GRID)
kpi_table['error_rate'] = error_rate.reindex(ALL_GRID)
kpi_table['ingestion_delay_proxy'] = ingestion_delay_proxy.reindex(ALL_GRID)
kpi_table['log_missing_rate'] = log_missing_rate.reindex(ALL_GRID)

# Analysis-only
kpi_table['request_count'] = request_count.reindex(ALL_GRID)
kpi_table['timeout_count'] = timeout_count.reindex(ALL_GRID)

# Label
kpi_table['incident_flag'] = incident_flag

print(f'kpi_table shape: {kpi_table.shape}')
print(f'결측 현황:')
print(kpi_table.isna().sum())

kpi_table = kpi_table.fillna(method='ffill', limit=5).fillna(0)
print(f'\n처리 후 결측: {kpi_table.isna().sum().sum()}')
display(kpi_table.head())

kpi_table.to_parquet(os.path.join(OUT, 'kpi_table.parquet'))
print(f'\n저장: kpi_table.parquet')

## Step H. Train/Test split (v3: ±10분 buffer + 길이 체크)

In [ ]:
FEATURE_COLS = ['p99_latency', 'timeout_rate', 'error_rate', 
                'ingestion_delay_proxy', 'log_missing_rate']

incident_buffer = (
    kpi_table['incident_flag']
    .rolling(window=2 * TRAIN_BUFFER_MIN + 1, center=True, min_periods=1)
    .max()
)

train_mask = (incident_buffer == 0)
train_df = kpi_table.loc[train_mask, FEATURE_COLS].copy()
test_df = kpi_table[FEATURE_COLS + ['incident_flag']].copy()

n_direct_incident = (kpi_table['incident_flag'] == 1).sum()
n_buffer_only = ((incident_buffer > 0) & (kpi_table['incident_flag'] == 0)).sum()
print(f'Train: {len(train_df)} 분 (정상 + buffer 제외)')
print(f'  - incident 직접 제외: {n_direct_incident} 분')
print(f'  - buffer (±{TRAIN_BUFFER_MIN}분) 제외: {n_buffer_only} 분')
print(f'Test: {len(test_df)} 분 (정상 {(test_df["incident_flag"]==0).sum()} + 이상 {(test_df["incident_flag"]==1).sum()})')

# v3: 안전장치 — train 데이터 너무 적으면 즉시 중단
if len(train_df) < MIN_TRAIN_MINUTES:
    raise ValueError(
        f'Train 데이터가 너무 적습니다: {len(train_df)}분 (최소 {MIN_TRAIN_MINUTES}분 필요).\n'
        f'해결책:\n'
        f'  1. Part 1에서 정상 비교용 날짜를 더 추가 (N_NORMAL_DAYS 늘리기)\n'
        f'  2. TRAIN_BUFFER_MIN 줄이기 (현재 {TRAIN_BUFFER_MIN}분)\n'
        f'  3. fault label에서 너무 자주 발생하는 fault 일부 제외 검토'
    )

## Step I. 정규화 + percentile metadata + risk-normalized table (v3 추가)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_df.values)

train_scaled = scaler.transform(train_df.values)
test_scaled = scaler.transform(test_df[FEATURE_COLS].values)

print(f'train scaled: {train_scaled.shape}')
print(f'test scaled:  {test_scaled.shape}')

# percentile (Part 4 트리거용, train 정상 분포)
train_percentiles = {}
for col in FEATURE_COLS:
    vals = train_df[col].dropna()
    train_percentiles[col] = {
        'p50': float(vals.quantile(0.50)),
        'p75': float(vals.quantile(0.75)),
        'p90': float(vals.quantile(0.90)),
        'p95': float(vals.quantile(0.95)),
        'p99': float(vals.quantile(0.99)),
    }

print(f'\n=== Train 정상 분포 percentile (Part 4 트리거용) ===')
display(pd.DataFrame(train_percentiles).T)

with open(os.path.join(OUT, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print(f'\n저장: scaler.pkl')

In [ ]:
# v3: risk-normalized table (Part 3 rule-based score용)
# 각 KPI를 train 정상 분포 P95로 나누어 0~1 정규화

risk_norm = pd.DataFrame(index=kpi_table.index)

for col in FEATURE_COLS:
    p95 = train_percentiles[col]['p95']
    if p95 == 0 or pd.isna(p95):
        # P95가 0이면 (예: ingestion_delay_proxy가 정상 시 0) → 다른 기준 사용
        # 차선책: 전체 max로 정규화, 그것도 0이면 0으로
        max_val = kpi_table[col].max()
        if max_val > 0:
            risk_norm[f'S_{col}'] = (kpi_table[col] / max_val).clip(0, 1)
        else:
            risk_norm[f'S_{col}'] = 0
    else:
        risk_norm[f'S_{col}'] = (kpi_table[col] / p95).clip(0, 1)

risk_norm['incident_flag'] = kpi_table['incident_flag']

print(f'risk_norm shape: {risk_norm.shape}')
print(f'\n각 S_{col} 통계 (정상이면 ≤ 0.95, fault 시 → 1.0):')
display(risk_norm.describe())

risk_norm.to_parquet(os.path.join(OUT, 'kpi_table_risk_normalized.parquet'))
print(f'\n저장: kpi_table_risk_normalized.parquet')
print(f'Part 3 사용 예시:')
print(f'  P_cloud_rule = 0.35*S_p99_latency + 0.25*S_timeout_rate + 0.20*S_error_rate + ...')

## Step J. 시퀀스 윈도우 생성 (v3: segment 기반)

**v3 변경**: train_df는 시간 불연속 구간 (incident buffer 제외)을 가짐. 연속된 정상 segment별로 sequence 생성하여 LSTM이 시간 점프를 정상 패턴으로 학습하지 않도록 함.

보고서: *"train sequence는 연속된 정상 구간(segment)별로 생성되며, incident buffer로 끊어진 구간은 별도 segment로 분리하여 시간적 불연속성이 모델 학습에 영향을 주지 않도록 하였다."*

In [ ]:
WINDOW_SIZE = 5

# v3: train segment 식별 (시간 차 1분 초과 = 새 segment)
train_with_idx = train_df.copy()
train_with_idx['time_diff_sec'] = (
    train_with_idx.index.to_series().diff().dt.total_seconds().fillna(60)
)
train_with_idx['segment_id'] = (train_with_idx['time_diff_sec'] > 60).cumsum()

# segment 통계
seg_sizes = train_with_idx.groupby('segment_id').size()
n_segments = len(seg_sizes)
n_usable_segments = (seg_sizes >= WINDOW_SIZE).sum()
n_dropped_minutes = seg_sizes[seg_sizes < WINDOW_SIZE].sum()

print(f'Train segment 통계:')
print(f'  전체 segment 수: {n_segments}')
print(f'  사용 가능 segment (≥{WINDOW_SIZE}분): {n_usable_segments}')
print(f'  너무 짧아 폐기되는 분 수: {n_dropped_minutes}')
print(f'\nsegment 길이 분포:')
print(f'  min: {seg_sizes.min()}, max: {seg_sizes.max()}, median: {int(seg_sizes.median())}')

In [ ]:
def create_sequences_by_segment(df_with_seg, scaled_array, feature_cols, window_size):
    """segment별로 sequence 생성 (시간 불연속 가로지르지 않음)"""
    sequences = []
    # df_with_seg와 scaled_array는 같은 row 순서를 가져야 함
    seg_ids = df_with_seg['segment_id'].values
    
    unique_segs = np.unique(seg_ids)
    for seg_id in unique_segs:
        seg_mask = seg_ids == seg_id
        seg_data = scaled_array[seg_mask]
        if len(seg_data) < window_size:
            continue
        for i in range(len(seg_data) - window_size + 1):
            sequences.append(seg_data[i:i+window_size])
    return np.array(sequences) if sequences else np.empty((0, window_size, len(feature_cols)))

# Train: segment 기반 (불연속 가로지르지 않음)
X_train = create_sequences_by_segment(train_with_idx, train_scaled, FEATURE_COLS, WINDOW_SIZE)

# Test: 연속 시간축이라 그냥 sliding window
def create_sequences_simple(data, window_size, labels=None):
    sequences = []
    seq_labels = []
    for i in range(len(data) - window_size + 1):
        sequences.append(data[i:i+window_size])
        if labels is not None:
            seq_labels.append(int(labels[i:i+window_size].any()))
    seqs = np.array(sequences)
    if labels is not None:
        return seqs, np.array(seq_labels)
    return seqs

X_test, y_test = create_sequences_simple(
    test_scaled, WINDOW_SIZE, labels=test_df['incident_flag'].values
)

print(f'X_train shape: {X_train.shape}  (segment 기반)')
print(f'X_test shape:  {X_test.shape}')
print(f'y_test fault 비율: {y_test.mean():.4f}')

# v3: X_train 안전장치
if len(X_train) < WINDOW_SIZE:
    raise ValueError(
        f'X_train sequence 너무 적음: {len(X_train)}개. '
        f'대부분 segment가 window_size({WINDOW_SIZE})보다 짧음. '
        f'TRAIN_BUFFER_MIN 줄이거나 정상 일자 추가 필요.'
    )

### J-검증. 저장 + Phase 3 인풋 준비

In [ ]:
np.save(os.path.join(OUT, 'X_train.npy'), X_train)
np.save(os.path.join(OUT, 'X_test.npy'), X_test)
np.save(os.path.join(OUT, 'y_test.npy'), y_test)

# v3: 풍부한 metadata
metadata = {
    # 기본 정보
    'feature_cols': FEATURE_COLS,
    'analysis_cols': ['request_count', 'timeout_count'],
    'label_col': 'incident_flag',
    'window_size': WINDOW_SIZE,
    'X_train_shape': list(X_train.shape),
    'X_test_shape': list(X_test.shape),
    'fault_ratio_in_test': float(y_test.mean()),
    
    # 임계치 / 파라미터
    'timeout_threshold_ms': TIMEOUT_THRESHOLD_MS,
    'train_buffer_min': TRAIN_BUFFER_MIN,
    'min_train_minutes': MIN_TRAIN_MINUTES,
    
    # fallback
    'used_fallback_for_error_rate': USED_FALLBACK_FOR_ERROR_RATE,
    
    # Part 4 보험 트리거용
    'train_normal_percentiles': train_percentiles,
    
    # v3: duration 분포 검증 결과
    'duration_quantiles_ms': DURATION_QUANTILES,
    
    # v3: segment 정보
    'train_segments': {
        'n_segments': int(n_segments),
        'n_usable_segments': int(n_usable_segments),
        'n_dropped_minutes': int(n_dropped_minutes),
        'min_segment_len': int(seg_sizes.min()),
        'max_segment_len': int(seg_sizes.max()),
        'median_segment_len': int(seg_sizes.median()),
    },
    
    # v3: proxy 변수 정의 (발표 방어용)
    'proxy_definitions': {
        'ingestion_delay_proxy': (
            '1 if no trace exists in the 1-minute slot, else 0. '
            'Used as telemetry availability gap proxy. '
            'Limitation: quiet periods (no requests) also marked as 1.'
        ),
        'log_missing_rate': (
            '1 - (actual business metric count / expected count). '
            'Expected = median of non-zero minutes. '
            'Proxy for monitoring log missing rate (not direct missing label).'
        ),
        'error_rate_fallback_used': USED_FALLBACK_FOR_ERROR_RATE,
        'error_rate_fallback_definition': (
            'When business success/error KPI not found, timeout_rate is used as proxy. '
            'Limitation: error_rate and timeout_rate carry redundant information in this case.'
        ),
    },
}
with open(os.path.join(OUT, 'phase2_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'저장 완료:')
print(f'  X_train.npy   {X_train.shape}')
print(f'  X_test.npy    {X_test.shape}')
print(f'  y_test.npy    {y_test.shape}')
print(f'  scaler.pkl')
print(f'  kpi_table.parquet (raw 8 컬럼)')
print(f'  kpi_table_risk_normalized.parquet (S_xxx, 0~1)')
print(f'  phase2_metadata.json (proxy 정의 + duration 분위수 + segment 정보 포함)')
print(f'\n→ Part 3 (LSTM Autoencoder) 진행 가능')

## ✅ Part 2 v3 완료 체크리스트

- [ ] Step A: 모든 컬럼 매핑 완료
- [ ] Step E-1: P99/counts/timeout 시계열 + duration 분포 검증 출력
- [ ] Step E-2: error rate (또는 fallback)
- [ ] Step E-3, E-4: proxy 산출
- [ ] Step F: incident flag 분포
- [ ] Step G: kpi_table.parquet 저장
- [ ] Step H: train_df 길이 충분 (≥ 100분)
- [ ] Step I: scaler + percentile + **risk-normalized table** 저장
- [ ] Step J: **segment 기반** sequence + 안전장치

## 출력 파일

| 파일 | 용도 |
|---|---|
| `kpi_table.parquet` | raw KPI (Part 3, Part 4 EDA용) |
| `kpi_table_risk_normalized.parquet` | **Part 3 rule-based score용** |
| `scaler.pkl` | StandardScaler (Part 3 LSTM 입력용) |
| `X_train.npy` / `X_test.npy` / `y_test.npy` | LSTM 학습/평가 입력 |
| `phase2_metadata.json` | percentile, duration 분위수, segment 정보, proxy 정의 |

## v2 → v3 변경사항 (발표 자료용)

1. **timeout threshold 분포 검증**: "5초 임계치가 데이터 P99/P999 분위수 기준 합리적 범위에 있음을 확인"
2. **proxy metadata 명시**: "ingestion_delay_proxy, log_missing_rate가 직접 측정값이 아님을 metadata에 명시적으로 정의"
3. **train_df 안전장치**: "학습 데이터가 100분 미만이면 즉시 중단하여 의미 없는 학습 방지"
4. **segment 기반 sequence**: "incident buffer로 끊어진 구간을 segment로 분리하여 LSTM이 시간 불연속을 정상 패턴으로 학습하지 않도록 함"
5. **error rate fallback 명시**: "business KPI 부재 시 timeout_rate를 proxy로 사용함을 metadata에 기록"
6. **risk-normalized table**: "Part 3 rule-based Cloud Risk Score 계산을 위한 0~1 정규화 KPI table 별도 저장"